# Mantener ejemplos legibles en docs y ejecutables en Binder

Este cuaderno documenta el flujo de sincronización: los scripts `.py` siguen siendo la versión explicativa y probada por MkDocs; Binder abre sus equivalentes `.ipynb`. En este repositorio los scripts canónicos viven en `examples/` (las páginas de docs ya los incluyen desde allí), y las notebooks se generan en `binder/notebooks/` para no mover ni duplicar la fuente.

## 1. Emparejar formatos con Jupytext

Instala Jupytext junto con el entorno de Binder:

```bash
pip install jupytext
```

La configuración de `binder/notebooks/jupytext.toml` declara que cada notebook empareja con el script de `examples/` con el mismo nombre:

```toml
formats = "ipynb,../../examples//py:percent"
```

Comprueba las rutas pares o sincroniza una notebook cuando cambie cualquiera de las dos versiones:

```bash
jupytext --paired-paths binder/notebooks/02_faults_and_coverage.ipynb
jupytext --sync binder/notebooks/02_faults_and_coverage.ipynb
```

In [1]:
from importlib.metadata import version

print("Jupytext", version("jupytext"))

Jupytext 1.19.5


## 2. Separar ejemplos de documentación y ejecución

En un proyecto nuevo se puede usar `docs/examples/*.py` como fuente legible y `binder/notebooks/*.ipynb` como interfaz interactiva. Aquí ya existe `examples/*.py`, referenciado por MkDocs y por `tests/examples`; lo conservamos como fuente canónica y emparejamos desde `binder/notebooks/` mediante prefijo Jupytext.

```text
examples/                 # scripts .py usados por la documentación y CI
binder/notebooks/         # notebooks .ipynb publicados por Binder
```

La comprobación siguiente crea el directorio de notebooks si falta y detecta cualquier ejemplo sin pareja en cualquiera de los dos lados.

In [2]:
from pathlib import Path

cwd = Path.cwd()
if (cwd / "examples").is_dir():
    repo_root = cwd
elif (cwd.parent.parent / "examples").is_dir():
    repo_root = cwd.parent.parent
else:
    raise FileNotFoundError("Run this notebook from a spicefault checkout")

script_dir = repo_root / "examples"
notebook_dir = repo_root / "binder" / "notebooks"
notebook_dir.mkdir(parents=True, exist_ok=True)
script_stems = {path.stem for path in script_dir.glob("0[1-7]_*.py")}
notebook_stems = {
    path.stem for path in notebook_dir.glob("*.ipynb") if not path.name.startswith("00_")
}
print("scripts without notebooks:", sorted(script_stems - notebook_stems))
print("notebooks without scripts:", sorted(notebook_stems - script_stems))

scripts without notebooks: []
notebooks without scripts: []


## 3. Crear un `.py` y sincronizarlo a notebook

Jupytext entiende celdas marcadas con `# %%` (código) y `# %% [markdown]` (texto). Los scripts actuales se conservan ejecutables desde terminal; añadir esos marcadores solo organiza las mismas secciones como celdas:

```python
# %% [markdown]
# # Faults and coverage
# Describe qué fallos cubre el universo.

# %%
from spicefault.faults import FaultSet, OpenCircuit

faults = FaultSet([OpenCircuit("R2")])
print(faults.ids())
```

Genera o sincroniza el notebook emparejado con Jupytext. En VS Code, abre el `.ipynb` de `binder/notebooks/` y ejecuta sus celdas de arriba abajo; `kernelspec` debe identificar Python 3.

In [3]:
import subprocess
import sys

example_notebook = notebook_dir / "02_faults_and_coverage.ipynb"
subprocess.run(
    [sys.executable, "-m", "jupytext", "--sync", str(example_notebook)],
    cwd=repo_root,
    check=True,
)
print("Synchronized:", example_notebook.relative_to(repo_root))

[jupytext] Reading /Users/telmomm/Desktop/spicefault/binder/notebooks/02_faults_and_coverage.ipynb in format ipynb
[jupytext] Unchanged /Users/telmomm/Desktop/spicefault/binder/notebooks/02_faults_and_coverage.ipynb
[jupytext] Unchanged /Users/telmomm/Desktop/spicefault/examples/02_faults_and_coverage.py
Synchronized: binder/notebooks/02_faults_and_coverage.ipynb


## 4. Automatizar la conversión masiva

`python scripts/sync_binder_examples.py` convierte de forma incremental todos los scripts numerados y actualiza sus parejas. `--check` solo informa de parejas ausentes o desactualizadas; `--to-scripts` sincroniza cambios hechos en notebooks de vuelta al `.py` canónico. El script no ejecuta las campañas: evita simulaciones inesperadas durante una conversión.

In [4]:
sync_script = repo_root / "scripts" / "sync_binder_examples.py"
result = subprocess.run(
    [sys.executable, str(sync_script), "--check"],
    cwd=repo_root,
    capture_output=True,
    text=True,
)
print(result.stdout or result.stderr)
result.check_returncode()

7 Binder notebooks are synchronised



## 5. Configurar Binder y enlaces directos

Binder necesita JupyterLab, Jupytext y las dependencias del proyecto; el `postBuild` instala `spicefault` desde el checkout y `binder/apt.txt` instala ngspice. Los notebooks se abren con un enlace que apunta a `binder/notebooks/<nombre>.ipynb`:

```text
https://mybinder.org/v2/gh/telmomm/spicefault/main?labpath=binder%2Fnotebooks%2F02_faults_and_coverage.ipynb
```

Cada cuaderno debe ejecutarse desde el checkout y cargar los datos relativos desde el repositorio; no debe depender de que se haya ejecutado antes el `.py`.

In [5]:
from urllib.parse import quote


def binder_link(notebook_path, branch="main"):
    return (
        "https://mybinder.org/v2/gh/telmomm/spicefault/"
        f"{branch}?labpath={quote(notebook_path)}"
    )


print(binder_link("binder/notebooks/02_faults_and_coverage.ipynb"))

https://mybinder.org/v2/gh/telmomm/spicefault/main?labpath=binder/notebooks/02_faults_and_coverage.ipynb


## 6. Validar en VS Code y CI

En un checkout local, instala `pip install -e ".[dev]"` para tener el kernel y el runner de notebooks. En VS Code abre `binder/notebooks/`, selecciona el kernel Python 3 y ejecuta las celdas en orden. Binder obtiene Jupyter/Jupytext de `binder/requirements.txt`, instala ngspice desde `binder/apt.txt` y el paquete desde `binder/postBuild`.

La prueba `tests/examples/test_examples.py` valida el esquema de todos los `.ipynb`, comprueba que siguen emparejados con sus scripts y ejecuta los notebooks 01 y 02 con `nbclient`. Los `.py` continúan siendo la versión que ejecuta el resto de pruebas y que se incluye en las páginas de documentación.

In [7]:
import nbformat
from nbclient import NotebookClient

example_notebooks = sorted(
    path for path in notebook_dir.glob("*.ipynb") if not path.name.startswith("00_")
)
for path in example_notebooks:
    candidate = nbformat.read(path, as_version=4)
    nbformat.validate(candidate)
    assert candidate.metadata.get("kernelspec", {}).get("name") == "python3"
print(f"Validated {len(example_notebooks)} notebook files and Python kernels")

smoke = nbformat.read(notebook_dir / "02_faults_and_coverage.ipynb", as_version=4)
NotebookClient(smoke, timeout=120, kernel_name="python3").execute(cwd=str(repo_root))
print("Notebook smoke execution passed: 02_faults_and_coverage.ipynb")

Validated 7 notebook files and Python kernels


[IPKernelApp] WARNING | Kernel is running over TCP without encryption. All communication (including code and outputs) is sent in plain text and is susceptible to eavesdropping. Use IPC transport or launch with kernel manager-provisioned CurveZMQ keys to enable transport encryption.


Notebook smoke execution passed: 02_faults_and_coverage.ipynb
